# Experimento: Data Splitting Bilingüe

Comparación de métodos de chunking sobre documentos en **español** y en **inglés**.


**Preguntas**
Decime la cantidad total de trabajadores contratados en los emprendimientos que a través del RIGI (Problema de tablas)

In [77]:
from pathlib import Path
from haystack import Document
from haystack.components.preprocessors import DocumentSplitter
import pandas as pd

## 1. Carga de documentos

In [61]:
outputs_dir = Path("outputs")

en_path = outputs_dir / "InformeIngles.md"
es_path = outputs_dir / "informe-145-hcdn-40-50_salida_docling.md"

doc_en = Document(
    content=en_path.read_text(encoding="utf-8"),
    meta={"lang": "en", "source": en_path.name}
)
doc_es = Document(
    content=es_path.read_text(encoding="utf-8"),
    meta={"lang": "es", "source": es_path.name}
)

print(f"EN — {len(doc_en.content.split())} palabras")
print(f"ES — {len(doc_es.content.split())} palabras")

EN — 270 palabras
ES — 4114 palabras


## 2. Static Splitting — por palabras

Usamos `DocumentSplitter` de HayStack con `split_by="word"`.

Parámetros:
- `split_length = 100` palabras por chunk
- `split_overlap = 15` palabras de solapamiento (~15%)

> Arrancamos con 100 palabras para que sea fácil de leer manualmente.

In [62]:
splitter = DocumentSplitter(
    split_by="word",
    split_length=100,
    split_overlap=15,
)
splitter.warm_up()

In [63]:
chunks_en = splitter.run(documents=[doc_en])["documents"]
chunks_es = splitter.run(documents=[doc_es])["documents"]

print(f"EN → {len(chunks_en)} chunks")
print(f"ES → {len(chunks_es)} chunks")

EN → 3 chunks
ES → 131 chunks


## 3. Inspección manual de chunks

In [65]:
def print_chunks(chunks: list, label: str) -> None:
    print(f"\n{'='*60}")
    print(f"  {label} — {len(chunks)} chunks")
    print(f"{'='*60}")
    for i, chunk in enumerate(chunks):
        word_count = len(chunk.content.split())
        print(f"\n--- Chunk {i+1} ({word_count} palabras) ---")
        print(chunk.content.strip())

print_chunks(chunks_en, "INGLÉS")
print_chunks(chunks_es, "ESPAÑOL")


  INGLÉS — 3 chunks

--- Chunk 1 (104 palabras) ---
# QUESTION No. 51
EMPLOYMENT, LABOR REGISTRATION AND RIGI
Report, according to records, the total number of workers hired in the projects that are being carried out through the RIGI. Provide details on the number and type of projects and their geographic location; for each one, indicate the number of workers hired and the average duration of employment.

## RESPONSE
The Ministry of Economy reports that the 12 projects approved within the framework of the RIGI, together with the Diablillos Project, which to date has a recommendation for approval by the RIGI Project Evaluation Committee and has been submitted for final resolution by the

--- Chunk 2 (104 palabras) ---
by the RIGI Project Evaluation Committee and has been submitted for final resolution by the Enforcement Authority, are projected to generate 36,873 jobs, including both direct and indirect employment, taking into account employment projections for both the construction an

## 5. Exportar chunks a `Chunks/`

In [66]:
chunks_dir = Path("Chunks")
chunks_dir.mkdir(exist_ok=True)

def save_chunks_md(chunks: list, lang: str, method: str = "static") -> None:
    lines = [f"# Chunks — method: {method} | lang: {lang}\n\n"]
    for chunk in chunks:
        m = chunk.meta
        word_count = len(chunk.content.split())
        overlap = m.get('_split_overlap', [{}])
        overlap_range = overlap[0].get('range', '') if overlap else ''
        lines.append(
            f"---\n"
            f"chunk_id: {m.get('split_id', '')}\n"
            f"source: {m.get('source', '')}\n"
            f"page_number: {m.get('page_number', '')}\n"
            f"split_idx_start: {m.get('split_idx_start', '')}\n"
            f"overlap_range: \"{overlap_range}\"\n"
            f"word_count: {word_count}\n"
            f"lang: {lang}\n"
            f"method: {method}\n"
            f"---\n\n"
        )
        lines.append(chunk.content.strip() + "\n\n")
    filename = chunks_dir / f"{method}_{lang}.md"
    filename.write_text("".join(lines), encoding="utf-8")
    print(f"Guardado: {filename} ({len(chunks)} chunks)")

save_chunks_md(chunks_en, "en")
save_chunks_md(chunks_es, "es")

Guardado: Chunks\static_en.md (3 chunks)
Guardado: Chunks\static_es.md (131 chunks)


## 4. Tabla resumen

## Parte 3 — Hierarchical Splitting

Usamos `HierarchicalNodeParser` de **LlamaIndex**.

Idea:
- Parsea el documento en varios niveles de granularidad (padre → hijo → nieto).
- Cada nodo guarda `parent_id` y `child_ids` → árbol navegable.
- En retrieval se indexan las **hojas** (chunks chicos). Si varios hijos del mismo padre matchean, un `AutoMergingRetriever` devuelve el **padre** (más contexto).

Parámetros:
- `chunk_sizes = [2048, 512, 128]` tokens — tres niveles.
- `chunk_overlap = 20` tokens.

> Tokens, NO palabras. Importante para comparar contra static (que mide palabras).

In [67]:
from llama_index.core import Document as LIDocument
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes, get_root_nodes

li_doc_en = LIDocument(text=doc_en.content, metadata={"lang": "en", "source": en_path.name})
li_doc_es = LIDocument(text=doc_es.content, metadata={"lang": "es", "source": es_path.name})

hier_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2048, 512, 128],
    chunk_overlap=20,
)

nodes_en = hier_parser.get_nodes_from_documents([li_doc_en])
nodes_es = hier_parser.get_nodes_from_documents([li_doc_es])

leaves_en, roots_en = get_leaf_nodes(nodes_en), get_root_nodes(nodes_en)
leaves_es, roots_es = get_leaf_nodes(nodes_es), get_root_nodes(nodes_es)

print(f"EN — total: {len(nodes_en)} | roots: {len(roots_en)} | leaves: {len(leaves_en)}")
print(f"ES — total: {len(nodes_es)} | roots: {len(roots_es)} | leaves: {len(leaves_es)}")

EN — total: 7 | roots: 1 | leaves: 5
ES — total: 147 | roots: 5 | leaves: 120


### 3.1 Inspección del árbol jerárquico

Cada nodo tiene `parent_node` y `child_nodes` (relaciones). Los niveles se identifican por el `chunk_size` con el que se generó cada nodo (`metadata['chunk_size']` si existe, o por la posición en la lista).

In [68]:
from collections import Counter

def node_level(n, leaf_ids: set, root_ids: set) -> str:
    if n.node_id in root_ids:
        return "root"
    if n.node_id in leaf_ids:
        return "leaf"
    return "mid"

def summarize(nodes, label):
    leaf_ids = {n.node_id for n in get_leaf_nodes(nodes)}
    root_ids = {n.node_id for n in get_root_nodes(nodes)}
    levels = Counter(node_level(n, leaf_ids, root_ids) for n in nodes)
    word_counts_by_level = {lv: [] for lv in ["root", "mid", "leaf"]}
    for n in nodes:
        lv = node_level(n, leaf_ids, root_ids)
        word_counts_by_level[lv].append(len(n.text.split()))

    print(f"\n=== {label} ===")
    print(f"Total nodos: {len(nodes)}")
    for lv in ["root", "mid", "leaf"]:
        wc = word_counts_by_level[lv]
        if wc:
            print(f"  {lv:5} → {len(wc):3} nodos | palabras: min={min(wc)} avg={sum(wc)//len(wc)} max={max(wc)}")

summarize(nodes_en, "INGLÉS")
summarize(nodes_es, "ESPAÑOL")


=== INGLÉS ===
Total nodos: 7
  root  →   1 nodos | palabras: min=270 avg=270 max=270
  mid   →   1 nodos | palabras: min=270 avg=270 max=270
  leaf  →   5 nodos | palabras: min=14 avg=54 max=72

=== ESPAÑOL ===
Total nodos: 147
  root  →   5 nodos | palabras: min=612 avg=823 max=983
  mid   →  22 nodos | palabras: min=30 avg=187 max=267
  leaf  → 120 nodos | palabras: min=2 avg=36 max=63


In [69]:
def peek_tree(nodes, label, max_per_level=2):
    from llama_index.core.schema import NodeRelationship
    leaf_ids = {n.node_id for n in get_leaf_nodes(nodes)}
    root_ids = {n.node_id for n in get_root_nodes(nodes)}
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    shown = {"root": 0, "mid": 0, "leaf": 0}
    for n in nodes:
        lv = node_level(n, leaf_ids, root_ids)
        if shown[lv] >= max_per_level:
            continue
        shown[lv] += 1
        parent = n.relationships.get(NodeRelationship.PARENT)
        children = n.relationships.get(NodeRelationship.CHILD) or []
        parent_id = parent.node_id[:8] if parent else "—"
        n_children = len(children) if isinstance(children, list) else 0
        print(f"\n[{lv}] id={n.node_id[:8]} | parent={parent_id} | children={n_children} | words={len(n.text.split())}")
        print(n.text.strip()[:300] + ("..." if len(n.text) > 300 else ""))

peek_tree(nodes_en, "INGLÉS")
peek_tree(nodes_es, "ESPAÑOL")


  INGLÉS

[root] id=4689b46b | parent=— | children=1 | words=270
# QUESTION No. 51
EMPLOYMENT, LABOR REGISTRATION AND RIGI
Report, according to records, the total number of workers hired in the projects that are being carried out through the RIGI. Provide details on the number and type of projects and their geographic location; for each one, indicate the number o...

[mid] id=5399d2f6 | parent=4689b46b | children=5 | words=270
# QUESTION No. 51
EMPLOYMENT, LABOR REGISTRATION AND RIGI
Report, according to records, the total number of workers hired in the projects that are being carried out through the RIGI. Provide details on the number and type of projects and their geographic location; for each one, indicate the number o...

[leaf] id=430045b5 | parent=5399d2f6 | children=0 | words=58
# QUESTION No. 51
EMPLOYMENT, LABOR REGISTRATION AND RIGI
Report, according to records, the total number of workers hired in the projects that are being carried out through the RIGI. Provide details on 

### 3.2 Exportar chunks jerárquicos a `Chunks/`

Guardamos TODOS los nodos (root + mid + leaf) con su relación padre-hijo en el frontmatter. Esto sirve después para reconstruir el árbol al cargar.

In [70]:
from llama_index.core.schema import NodeRelationship, MetadataMode

def save_hier_chunks_md(nodes, lang: str, method: str = "hierarchical") -> None:
    leaf_ids = {n.node_id for n in get_leaf_nodes(nodes)}
    root_ids = {n.node_id for n in get_root_nodes(nodes)}
    lines = [f"# Chunks — method: {method} | lang: {lang}\n\n"]
    for n in nodes:
        parent = n.relationships.get(NodeRelationship.PARENT)
        children = n.relationships.get(NodeRelationship.CHILD) or []
        parent_id = parent.node_id if parent else ""
        child_ids = [c.node_id for c in children] if isinstance(children, list) else []
        lv = node_level(n, leaf_ids, root_ids)
        word_count = len(n.text.split())
        # Embed-mode content: lo que el embedder realmente recibe (metadata + texto)
        embed_content = n.get_content(metadata_mode=MetadataMode.EMBED).strip()
        lines.append(
            f"---\n"
            f"chunk_id: {n.node_id}\n"
            f"parent_id: {parent_id}\n"
            f"child_ids: {child_ids}\n"
            f"level: {lv}\n"
            f"word_count: {word_count}\n"
            f"lang: {lang}\n"
            f"method: {method}\n"
            f"source: {n.metadata.get('source', '')}\n"
            f"---\n\n"
            f"#### [EMBED VIEW — lo que ve el embedder]\n\n"
            f"{embed_content}\n\n"
            f"#### [RAW TEXT — solo el contenido]\n\n"
            f"{n.text.strip()}\n\n"
        )
    filename = chunks_dir / f"{method}_{lang}.md"
    filename.write_text("".join(lines), encoding="utf-8")
    print(f"Guardado: {filename} ({len(nodes)} nodos)")

save_hier_chunks_md(nodes_en, "en")
save_hier_chunks_md(nodes_es, "es")

Guardado: Chunks\hierarchical_en.md (7 nodos)
Guardado: Chunks\hierarchical_es.md (147 nodos)


### 3.3 Tabla comparativa — hierarchical vs static

Mismo formato que la tabla anterior, pero ahora con la columna `level` (root / mid / leaf).

In [ ]:
def hier_nodes_to_df(nodes, lang: str) -> pd.DataFrame:
    leaf_ids = {n.node_id for n in get_leaf_nodes(nodes)}
    root_ids = {n.node_id for n in get_root_nodes(nodes)}
    return pd.DataFrame([
        {
            "lang": lang,
            "level": node_level(n, leaf_ids, root_ids),
            "chunk_idx": i,
            "word_count": len(n.text.split()),
            "preview": n.text.strip()[:120].replace("\n", " ") + "...",
        }
        for i, n in enumerate(nodes)
    ])

df_hier = pd.concat(
    [hier_nodes_to_df(nodes_en, "en"), hier_nodes_to_df(nodes_es, "es")],
    ignore_index=True
)
df_hier.groupby(["lang", "level"]).agg(
    n=("chunk_idx", "count"),
    avg_words=("word_count", "mean"),
    min_words=("word_count", "min"),
    max_words=("word_count", "max"),
).round(1)

## Parte 4 — Markdown-aware Splitting

Dos variantes:

- **4.A `MarkdownNodeParser` puro** — corta SOLO por headers (`#`, `##`, `###`). NO cuenta tokens. Una sección = un nodo. Respeta tablas si caen adentro de una sección.
- **4.B Híbrido `MarkdownNodeParser` + `SentenceSplitter`** — primero corta por headers, después subdivide los nodos grandes por tokens. Lo mejor de ambos.

Metadata: `MarkdownNodeParser` agrega `header_path` (tipo `H1 / H2 / H3`) → trazabilidad estructural sin árbol.

In [71]:
import re

# === 1. Buscar caracteres ocultos en los headers ===
print("=== CHARS RAROS ===")
for i, line in enumerate(doc_es.content.split("\n")[:80]):
    if "PREGUNTA" in line.upper() or "RESPUESTA" in line.upper():
        print(f"line {i}: {repr(line)}")

# === 2. Ver metadata REAL de los nodos de MarkdownNodeParser ===
print("\n=== METADATA REAL md_nodes_es ===")
for i, n in enumerate(md_nodes_es[:8]):
    print(f"--- node {i} | words={len(n.text.split())} ---")
    print(f"  metadata keys: {list(n.metadata.keys())}")
    print(f"  metadata: {n.metadata}")
    print(f"  text starts: {n.text[:100]!r}")
    print()

  # === 3. Versión instalada ===
import llama_index.core
print(f"\nllama-index-core version: {llama_index.core.__version__}")

=== CHARS RAROS ===
line 0: '## PREGUNTA N° 43'
line 4: '## RESPUESTA'
line 10: '## PREGUNTA N° 44'
line 14: '## RESPUESTA'
line 20: 'Para más información puede consultarla respuestaa la Pregunta N° 1416 del presente Informe.'
line 22: '## PREGUNTA N° 45'
line 26: '## RESPUESTA'
line 46: '## PREGUNTA N° 46'
line 50: '## RESPUESTA'
line 58: '## PREGUNTA N° 47'
line 62: '## RESPUESTA'
line 72: 'generalistas/guardia y personal de salud mental en el ámbito del Servicio Penitenciario Federal; se efectuaron capacitaciones formales para los profesionalesde los establecimientospenitenciarios,fortaleciendosu capacidad de respuesta y estrategias de abordaje en las diferentes situaciones que pudieran presentarse, entre otras.'

=== METADATA REAL md_nodes_es ===
--- node 0 | words=179 ---
  metadata keys: ['lang', 'source', 'header_path']
  metadata: {'lang': 'es', 'source': 'informe-145-hcdn-40-50_salida_docling.md', 'header_path': '/'}
  text starts: '## PREGUNTA N° 43\n\nDEFENSANACIONAL -F-16 Y

### 4.A `MarkdownNodeParser` puro

In [72]:
from llama_index.core.node_parser import MarkdownNodeParser

md_parser = MarkdownNodeParser()

md_nodes_en = md_parser.get_nodes_from_documents([li_doc_en])
md_nodes_es = md_parser.get_nodes_from_documents([li_doc_es])

print(f"EN — {len(md_nodes_en)} secciones")
print(f"ES — {len(md_nodes_es)} secciones")

EN — 2 secciones
ES — 29 secciones


In [73]:
def peek_md(nodes, label, max_show=3):
    print(f"\n{'='*60}\n  {label} — {len(nodes)} secciones\n{'='*60}")
    for i, n in enumerate(nodes[:max_show]):
        header_path = " / ".join(
            v for k, v in n.metadata.items() if k.lower().startswith("header_")
        )
        wc = len(n.text.split())
        print(f"\n--- Sección {i+1} | words={wc} | header_path: {header_path or '(sin header)'} ---")
        print(n.text.strip()[:300] + ("..." if len(n.text) > 300 else ""))

peek_md(md_nodes_en, "INGLÉS")
peek_md(md_nodes_es, "ESPAÑOL")


  INGLÉS — 2 secciones

--- Sección 1 | words=58 | header_path: / ---
# QUESTION No. 51
EMPLOYMENT, LABOR REGISTRATION AND RIGI
Report, according to records, the total number of workers hired in the projects that are being carried out through the RIGI. Provide details on the number and type of projects and their geographic location; for each one, indicate the number o...

--- Sección 2 | words=212 | header_path: /QUESTION No. 51/ ---
## RESPONSE
The Ministry of Economy reports that the 12 projects approved within the framework of the RIGI, together with the Diablillos Project, which to date has a recommendation for approval by the RIGI Project Evaluation Committee and has been submitted for final resolution by the Enforcement Au...

  ESPAÑOL — 29 secciones

--- Sección 1 | words=179 | header_path: / ---
## PREGUNTA N° 43

DEFENSANACIONAL -F-16 Y FONDEF Informesi el Acuerdode Transferencia de los F-16 incluye cláusulas de restricción geográfica de despliegue o específicas limitaciones

### 4.B Híbrido — `MarkdownNodeParser` + `SentenceSplitter`

Pipeline en 2 pasos:
1. `MarkdownNodeParser` corta por headers → preserva estructura y tablas.
2. `SentenceSplitter(chunk_size=512, chunk_overlap=50)` subdivide secciones grandes → controla tamaño.

Resultado: chunks acotados en tokens PERO con `header_path` heredado del padre.

In [74]:
from llama_index.core.node_parser import SentenceSplitter

sentence_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

hybrid_nodes_en = sentence_splitter.get_nodes_from_documents(md_nodes_en)
hybrid_nodes_es = sentence_splitter.get_nodes_from_documents(md_nodes_es)

print(f"EN — {len(md_nodes_en)} secciones → {len(hybrid_nodes_en)} chunks")
print(f"ES — {len(md_nodes_es)} secciones → {len(hybrid_nodes_es)} chunks")

EN — 2 secciones → 2 chunks
ES — 29 secciones → 36 chunks


In [56]:
peek_md(hybrid_nodes_en, "INGLÉS (híbrido)")
peek_md(hybrid_nodes_es, "ESPAÑOL (híbrido)")


  INGLÉS (híbrido) — 2 secciones

--- Sección 1 | words=58 | header_path: / ---
# QUESTION No. 51
EMPLOYMENT, LABOR REGISTRATION AND RIGI
Report, according to records, the total number of workers hired in the projects that are being carried out through the RIGI. Provide details on the number and type of projects and their geographic location; for each one, indicate the number o...

--- Sección 2 | words=212 | header_path: / ---
# RESPONSE
The Ministry of Economy reports that the 12 projects approved within the framework of the RIGI, together with the Diablillos Project, which to date has a recommendation for approval by the RIGI Project Evaluation Committee and has been submitted for final resolution by the Enforcement Aut...

  ESPAÑOL (híbrido) — 36 secciones

--- Sección 1 | words=179 | header_path: / ---
## PREGUNTA N° 43

DEFENSANACIONAL -F-16 Y FONDEF Informesi el Acuerdode Transferencia de los F-16 incluye cláusulas de restricción geográfica de despliegue o específicas limitaci

### 4.C Exportar a `Chunks/`

Guardamos los dos: `markdown_{lang}.md` (puro) y `hybrid_{lang}.md` (con SentenceSplitter encima).

In [ ]:
from llama_index.core.schema import MetadataMode

def save_md_chunks(nodes, lang: str, method: str) -> None:
    lines = [f"# Chunks — method: {method} | lang: {lang}\n\n"]
    for i, n in enumerate(nodes):
        header_path = " / ".join(
            v for k, v in n.metadata.items() if k.lower().startswith("header_")
        )
        wc = len(n.text.split())
        # Embed-mode content: lo que el embedder realmente recibe (metadata + texto)
        embed_content = n.get_content(metadata_mode=MetadataMode.EMBED).strip()
        lines.append(
            f"---\n"
            f"chunk_id: {n.node_id}\n"
            f"chunk_idx: {i}\n"
            f"header_path: \"{header_path}\"\n"
            f"word_count: {wc}\n"
            f"lang: {lang}\n"
            f"method: {method}\n"
            f"source: {n.metadata.get('source', '')}\n"
            f"---\n\n"
            f"#### [EMBED VIEW — lo que ve el embedder]\n\n"
            f"{embed_content}\n\n"
            f"#### [RAW TEXT — solo el contenido]\n\n"
            f"{n.text.strip()}\n\n"
        )
    filename = chunks_dir / f"{method}_{lang}.md"
    filename.write_text("".join(lines), encoding="utf-8")
    print(f"Guardado: {filename} ({len(nodes)} chunks)")

#save_md_chunks(md_nodes_en, "en", "markdown")
#save_md_chunks(md_nodes_es, "es", "markdown")
save_md_chunks(hybrid_nodes_en, "en", "hybrid")
save_md_chunks(hybrid_nodes_es, "es", "hybrid")

Guardado: Chunks\markdown_en.md (2 chunks)
Guardado: Chunks\markdown_es.md (29 chunks)
Guardado: Chunks\hybrid_en.md (2 chunks)
Guardado: Chunks\hybrid_es.md (36 chunks)


### 4.D Comparativa final — static vs hierarchical vs markdown vs hybrid

In [79]:
def stats_haystack(chunks, lang, method):
    wc = [len(c.content.split()) for c in chunks]
    return {"method": method, "lang": lang, "n": len(wc),
            "min": min(wc), "avg": round(sum(wc)/len(wc), 1), "max": max(wc)}

def stats_llamaindex(nodes, lang, method):
    wc = [len(n.text.split()) for n in nodes]
    return {"method": method, "lang": lang, "n": len(wc),
            "min": min(wc), "avg": round(sum(wc)/len(wc), 1), "max": max(wc)}

rows = [
    stats_haystack(chunks_en, "en", "static"),
    stats_haystack(chunks_es, "es", "static"),
    stats_llamaindex(get_leaf_nodes(nodes_en), "en", "hierarchical (leaves)"),
    stats_llamaindex(get_leaf_nodes(nodes_es), "es", "hierarchical (leaves)"),
    stats_llamaindex(md_nodes_en, "en", "markdown"),
    stats_llamaindex(md_nodes_es, "es", "markdown"),
    stats_llamaindex(hybrid_nodes_en, "en", "hybrid"),
    stats_llamaindex(hybrid_nodes_es, "es", "hybrid"),
]

pd.DataFrame(rows).sort_values(["lang", "method"]).reset_index(drop=True)

,method,lang,n,min,avg,max
0,hierarchical (leaves),en,5,14,54.0,72
1,hybrid,en,2,58,135.0,212
2,markdown,en,2,58,135.0,212
3,static,en,3,94,100.7,104
4,hierarchical (leaves),es,120,2,36.4,63
5,hybrid,es,36,16,116.4,254
6,markdown,es,29,16,141.9,866
7,static,es,131,0,37.0,106


## Parte 5 — LangChain `MarkdownHeaderTextSplitter`

Librería estándar para chunking markdown. Hace lo que `MarkdownNodeParser` de LlamaIndex v0.14.22 está fallando:

- Detecta headers ATX correctamente.
- Setea `Header_1`, `Header_2`, etc. en metadata.
- `strip_headers=True` saca el `## RESPUESTA` del body → queda SOLO en metadata.
- Se combina nativo con `RecursiveCharacterTextSplitter` para subdividir secciones grandes (los sub-chunks **heredan** la metadata).

**Pipeline:**
1. `MarkdownHeaderTextSplitter` → secciones con headers en metadata.
2. `RecursiveCharacterTextSplitter` → subdivide por chars preservando metadata.
3. Convertimos a `LIDocument` para usar el mismo `save_md_chunks` y comparar lado a lado.

### 5.A Split por headers (LangChain)

Definimos qué niveles de header mapear y a qué key de metadata. `strip_headers=True` saca la línea del header del body.

In [34]:
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

headers_to_split_on = [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3"),
]

md_header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=True,
)

lc_sections_en = md_header_splitter.split_text(doc_en.content)
lc_sections_es = md_header_splitter.split_text(doc_es.content)

# Agregar source/lang manualmente (LangChain solo mete headers)
for d in lc_sections_en:
    d.metadata.update({"lang": "en", "source": en_path.name})
for d in lc_sections_es:
    d.metadata.update({"lang": "es", "source": es_path.name})

print(f"EN — {len(lc_sections_en)} secciones")
print(f"ES — {len(lc_sections_es)} secciones")
print()
print("=== Sample metadata (ES, primeras 3 secciones) ===")
for i, d in enumerate(lc_sections_es[:3]):
    print(f"\n--- sección {i} | words={len(d.page_content.split())} ---")
    print(f"  metadata: {d.metadata}")
    print(f"  text starts: {d.page_content[:120]!r}")

EN — 2 secciones
ES — 29 secciones

=== Sample metadata (ES, primeras 3 secciones) ===

--- sección 0 | words=175 ---
  metadata: {'Header_2': 'PREGUNTA N° 43', 'lang': 'es', 'source': 'informe-145-hcdn-40-50_salida_docling.md'}
  text starts: 'DEFENSANACIONAL -F-16 Y FONDEF Informesi el Acuerdode Transferencia de los F-16 incluye cláusulas de restricción geográf'

--- sección 1 | words=105 ---
  metadata: {'Header_2': 'RESPUESTA', 'lang': 'es', 'source': 'informe-145-hcdn-40-50_salida_docling.md'}
  text starts: 'El Ministeriode Defensainformaque el PoderEjecutivoNacionalha evaluadoel Acuerdode Transferenciade los F-16 en el marcod'

--- sección 2 | words=81 ---
  metadata: {'Header_2': 'PREGUNTA N° 44', 'lang': 'es', 'source': 'informe-145-hcdn-40-50_salida_docling.md'}
  text starts: 'DEFENSA NACIONAL -F-16 Y FONDEF Informe el porcentajede ejecución realdel FONDEF correspondienteal ejercicio2025 y lo pr'


### 5.B Subdivisión por tamaño (RecursiveCharacterTextSplitter)

Para secciones grandes — corta por chars manteniendo la metadata de la sección padre.

> Nota: LangChain `RecursiveCharacterTextSplitter` mide en **chars**, no tokens. Para tokens existe `TokenTextSplitter`. Acá usamos chars por simplicidad — `chunk_size=2000` ≈ 400-500 palabras.

In [35]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
)

lc_chunks_en = recursive_splitter.split_documents(lc_sections_en)
lc_chunks_es = recursive_splitter.split_documents(lc_sections_es)

print(f"EN — {len(lc_sections_en)} secciones → {len(lc_chunks_en)} chunks")
print(f"ES — {len(lc_sections_es)} secciones → {len(lc_chunks_es)} chunks")
print()
print("=== Verificación de herencia (ES, primeros 3 chunks) ===")
for i, d in enumerate(lc_chunks_es[:3]):
    print(f"\n--- chunk {i} | words={len(d.page_content.split())} ---")
    print(f"  metadata: {d.metadata}")
    print(f"  text starts: {d.page_content[:100]!r}")

EN — 2 secciones → 2 chunks
ES — 29 secciones → 37 chunks

=== Verificación de herencia (ES, primeros 3 chunks) ===

--- chunk 0 | words=175 ---
  metadata: {'Header_2': 'PREGUNTA N° 43', 'lang': 'es', 'source': 'informe-145-hcdn-40-50_salida_docling.md'}
  text starts: 'DEFENSANACIONAL -F-16 Y FONDEF Informesi el Acuerdode Transferencia de los F-16 incluye cláusulas de'

--- chunk 1 | words=105 ---
  metadata: {'Header_2': 'RESPUESTA', 'lang': 'es', 'source': 'informe-145-hcdn-40-50_salida_docling.md'}
  text starts: 'El Ministeriode Defensainformaque el PoderEjecutivoNacionalha evaluadoel Acuerdode Transferenciade l'

--- chunk 2 | words=81 ---
  metadata: {'Header_2': 'PREGUNTA N° 44', 'lang': 'es', 'source': 'informe-145-hcdn-40-50_salida_docling.md'}
  text starts: 'DEFENSA NACIONAL -F-16 Y FONDEF Informe el porcentajede ejecución realdel FONDEF correspondienteal e'


### 5.C Conversión a LlamaIndex + exportar

Convertimos `langchain.Document` → `LIDocument` para reutilizar `save_md_chunks`. Así el `.md` de salida queda con el mismo formato que las otras partes y la comparación es directa.

In [ ]:
def lc_to_li(lc_docs: list) -> list:
    """Convierte langchain.Document → LIDocument preservando metadata."""
    out = []
    for d in lc_docs:
        # header_path armado a partir de Header_*
        path_parts = [
            v for k, v in sorted(d.metadata.items()) if k.startswith("Header_")
        ]
        meta = dict(d.metadata)
        meta["header_path"] = "/" + "/".join(path_parts) if path_parts else "/"
        out.append(LIDocument(text=d.page_content, metadata=meta))
    return out

li_lc_chunks_en = lc_to_li(lc_chunks_en)
li_lc_chunks_es = lc_to_li(lc_chunks_es)


save_md_chunks(li_lc_chunks_en, "en", "langchain_hybrid")
save_md_chunks(li_lc_chunks_es, "es", "langchain_hybrid")

Guardado: Chunks\langchain_sections_en.md (2 chunks)
Guardado: Chunks\langchain_sections_es.md (29 chunks)
Guardado: Chunks\langchain_hybrid_en.md (2 chunks)
Guardado: Chunks\langchain_hybrid_es.md (37 chunks)


### 5.D Comparativa final — todos los métodos

In [37]:
rows = [
    stats_haystack(chunks_en, "en", "static"),
    stats_haystack(chunks_es, "es", "static"),
    stats_llamaindex(get_leaf_nodes(nodes_en), "en", "hierarchical (leaves)"),
    stats_llamaindex(get_leaf_nodes(nodes_es), "es", "hierarchical (leaves)"),
    stats_llamaindex(md_nodes_en, "en", "llamaindex_markdown"),
    stats_llamaindex(md_nodes_es, "es", "llamaindex_markdown"),
    stats_llamaindex(hybrid_nodes_en, "en", "llamaindex_hybrid"),
    stats_llamaindex(hybrid_nodes_es, "es", "llamaindex_hybrid"),
    stats_llamaindex(li_lc_chunks_en, "en", "langchain_hybrid"),
    stats_llamaindex(li_lc_chunks_es, "es", "langchain_hybrid"),
]

pd.DataFrame(rows).sort_values(["lang", "method"]).reset_index(drop=True)

NameError: name 'pd' is not defined